# SPARC Building Scan

The SPARC building scan is designed to extract and attribute building footprints for an entire city following a multistep process. To begin, we extract building footprints in the follow order:

1. Download buildings from Overture maps  
2. Evaluate completeness of the downloaded building footprints through comparison to other built area data:  
   a. World Settlement Footprint  
   b. GHSL  
   c. [POTENTIALLY] Alpha Earth embeddings
3. Identify areas for infill from #2; perform infill using additional data sources:  
   a. HOT OSM tasking  
   b. Purchase satellite imagery and run building extraction locally  
   c. Purchase extracted building footprints from commercial providers  

This notebook will focus on #1 and #2, with #3 being addressed in subsequent steps depending on the solution.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
from rasterio.mask import mask
from shapely.geometry import mapping, shape

sys.path.append("../src")
from building_coverage_helper import *

# auto reload modules
%load_ext autoreload
%autoreload 2

In [2]:
# Open the local city extent
CITY_EXTENTS_PATH = r"C:\WBG\Work\Projects\SPARC\building_scan\testing\CIV_UCDB_cities.geojson"
CITY_NAME_FIELD = "GC_UCN_MAI_2025"  # Update to a valid column in your extents file.

city_extents = gpd.read_file(CITY_EXTENTS_PATH)
city_extents = city_extents.to_crs(4326)

# Define output folder and files
OUTPUT_ROOT = Path(r"C:\WBG\Work\Projects\SPARC\building_scan\testing\output")
cities_folder = OUTPUT_ROOT / "cities"

for out_dir in [OUTPUT_ROOT, cities_folder]:
    Path(out_dir).mkdir(parents=True, exist_ok=True)

In [4]:
for idx, row in city_extents.iterrows():
    city_name = row[CITY_NAME_FIELD]
    out_folder = cities_folder / city_name
    city_processor = SPARC_city(city_name, row, out_folder, city_extents.crs)
    city_processor.download_overture()
    city_processor.download_wsf()
    city_processor.compare_overture_wsf()
    break

# DEBURGGING